# DIPLOME UNIVERSITAIRE – Sorbonne Data Analytics  
## Projet Python avancé – Analyse de marché immobilier

**Promotion :** 007  
**Session :** Décembre 2025  

---

## Sujet

> **Analyse du marché immobilier à partir de données web scraping**  
> Périmètre : secteur de **Bordeaux**, rayon **10 km** (R10), maisons exclusivement \
> `https://www.etreproprio.com/annonces/th.lc33063-r10.odd#list`

**Problématique :**  
> _Comment le prix au mètre carré varie-t-il en fonction de la localisation, de la surface et du type de bien immobilier en France (dans le secteur choisi) ?_


### Contexte et périmètre d’analyse

Le jeu de données utilisé dans ce projet est issu d’un web scraping du site EtreProprio.com sur un périmètre volontairement ciblé :
- **type de bien** : uniquement des **maisons** (pas d’appartements, la typologie sera portée sur les équipements complémentaires) ;
- **zone géographique** : secteur de **Bordeaux** et communes voisines dans un rayon de **10 km** (R10) ;
- **période** : un **snapshot** du marché au moment du scraping, sans historique temporel.

Le **type de bien** est filtré directement via le site Internet en amont sur les maisons, ce qui garantit une certaine homogénéité d’usage (habitat individuel). Des variables complémentaires comme `surface_terrain`, `features` (piscine, jardin, terrasse, parking…) et les diagnostics `dpe` / `ges` permettent d’étudier plus finement le positionnement prix/m² de certains segments (biens avec piscine, grands terrains, bonne ou mauvaise performance énergétique, etc.).

L’ensemble forme ainsi un instantané cohérent du marché des maisons autour de Bordeaux, suffisant pour analyser les prix au mètre carré en fonction de la localisation, de la surface et des caractéristiques des biens, tout en gardant en tête les limites liées au caractère ponctuel et partiellement incomplet des annonces.

Note : Ces données scrapées à un instant donné constituent une image du marché des maisons autour de Bordeaux (rayon 10 km), sans historique temporel.

---

## Pipeline du projet

1. **SCRAPING** – `SRC/SCRAPER.py`  
   - Script python : `SRC/SCRAPER.py`
   - Source : EtreProprio.com – maisons, Bordeaux R10  
   - Sortie CSV : `OUTPUT/data_bordeaux_R10_maisons.csv` (dossier de sortie pour le professeur)  
2. **NETTOYAGE & STRUCTURATION** – _ce notebook_  
   - Entrée CSV : `DATA/data_bordeaux_R10_maisons.csv` (version figée du brut)  
   - Sortie principale CSV : `DATA/data_bordeaux_R10_maisons_CLEAN.csv`
   - Script python associé : `SRC/CLEAN_DATA.py`
3. **EXPLORATION & ANALYSES** – _ce notebook_  
   - Entrée CSV : `data_bordeaux_R10_maisons_CLEAN.csv`  
   - Sorties : graphiques, tableaux, interprétations pour le rapport + application Streamlit `SRC/DASHBOARD.py`  
   - Script python associé : `SRC/ANALYSE.py`
4. **DASHBOARD INTERACTIF** (Streamlit) – `SRC/DASHBOARD.py`  
   - Entrée CSV : fichier CLEAN, enrichi des coordonnées géographiques
   - Script python : `SRC/DASHBOARD.py`

---

## Objectif

L’objectif de ce notebook est double : 

1. **Construire une base de données propre et documentée** (CLEAN), à partir du brut issu du scraping ;  
2. **Explorer et interpréter le marché des maisons autour de Bordeaux**, à travers le prisme du prix au m², de la localisation et des caractéristiques des biens, en vue d’un futur dashboard interactif.

Nous allons donc passer par les étapes suivantes :
1. **Nettoyage & structuration**  
   - Diagnostic du fichier brut (`df_raw`)  
   - Construction d’une base propre (`df_clean`, `df_final`)  
   - Enrichissement : géolocalisation (Latitude/Longitude) pour la cartographie
   - Export du fichier CLEAN officiel

2. **Exploration & analyses**  
   - Statistiques descriptives  
   - Prix/m² moyen et médian  
   - Effets de la localisation
   - Effets des features (piscine, jardin…)
   - Effets des diagnostics DPE/GES  
   - Identification et interprétation des outliers
   - Audit des valeurs manquantes

3. **Préparation géolocalisation & dashboard**  
   - Géocodage `(ville, cp) → (lat, lon)` via Nominatim  
   - Base exploitable pour **folium** et pour le **dashboard Streamlit**

---

#### Encadré technique – Fonction `extract_number` et expressions régulières
#### `re.sub` vs `re.findall` : pourquoi ces choix ?

Nous avons choisi d'implémenter directement dans le script du scraping (`SCRAPER.py`) la fonction `extract_number', pour travailler en amont sur les **expressions régulières** (`regex`) du module `re` et extraire proprement les valeurs numériques à partir de chaînes de caractères brutes.

Typiquement, `extract_number` utilise une regex de type :
- soit `re.sub(r"[^0-9]", "", texte)` pour supprimer tout ce qui n’est pas chiffre, => c'est la méthode utilisée, expliquée ci-dessous.
- soit `re.findall(r"[0-9]+", texte)` pour récupérer directement les blocs de chiffres.

Exemples de formats rencontrés dans le HTML :
- `"345 000 €"` (prix avec espace et symbole),
- `"120 m²"` (surface avec unité),
- `"5 pièces"` (nombre de pièces),
- `"Pessac 36000"` (code postal et ville dans le même bloc HTML).

C'est donc la structure du code HTML de la ville et du code postal qui a imposé le choix d'appliquer un regex `re.sub' au moment du scraping pour en extraire la partie numérique. Ces informations obligatoires étaient encapsulées dans une seule et même classe '< div class="ep-loc" > —&nbsp;Pessac&nbsp;33600&nbsp;— < /div >'.

Nous avions donc l'obligation de :
1. **Nettoyer** la chaîne de tout ce qui n'était pas un chiffre (espaces, unité, symbole `€`, etc.) ;
2. **Récupérer la partie numérique** et la convertir ensuite en `int` ou `float`.

Cette approche permet de rendre le scraping plus robuste face à :
- des variations de format (espaces, séparateurs, symboles),
- des textes plus longs où le nombre est noyé au milieu d’autres informations.

Les valeurs extraites sont ensuite converties en numérique dans le notebook via `pd.to_numeric`, ce qui garantit la cohérence des types pour l’analyse.

---

## **Mise en place**

In [ ]:
# Imports

import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# pour afficher les graphiques dans le notebook
%matplotlib inline  

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Timeout plus large (par ex. 5 secondes)
geolocator = Nominatim(user_agent="m1_immo_bordeaux", timeout=5)


In [ ]:
# Définition des chemins

BASE_DIR = os.getcwd()  # racine du projet, là où se trouve le notebook
DATA_DIR = os.path.join(BASE_DIR, "DATA")
OUTPUT_DIR = os.path.join(BASE_DIR, "OUTPUT")

# Fichier brut de référence POUR L'ANALYSE (figé)
RAW_CSV_PATH = os.path.join(DATA_DIR, "data_bordeaux_R10_maisons.csv")

# Fichier CLEAN officiel (résultat de ce notebook)
CLEAN_CSV_PATH = os.path.join(DATA_DIR, "data_bordeaux_R10_maisons_CLEAN.csv")

print("Répertoire de base :", BASE_DIR)
print("Répertoire DATA    :", DATA_DIR)
print("Répertoire OUTPUT  :", OUTPUT_DIR)
print("CSV brut (analyse) :", RAW_CSV_PATH)
print("CSV CLEAN (sortie) :", CLEAN_CSV_PATH)


**Remarque sur les fichiers CSV**

- Le **scraper** (`SCRAPER.PY`) écrit par défaut dans :  
  `OUTPUT/data_bordeaux_R10_maisons.csv`  
  → c’est la preuve de fonctionnement du scraping, que le professeur peut relancer.

- Pour le **travail d’analyse**, on fige une version de ce brut dans :  
  `DATA/data_bordeaux_R10_maisons.csv`  
  → c’est ce fichier qui sert de **référence** pour tout le nettoyage et les analyses.  
  → ainsi, relancer le scraper ne vient jamais écraser la base avec laquelle on travaille ici.

## 1. **Nettoyage & structuration**

### 1.1. **Diagnostic**

In [ ]:
# 1.1.1. Chargement du brut (df_raw) + premier diagnostic

# Vérification de l'existence du fichier brut
assert os.path.exists(RAW_CSV_PATH), f"❌ Fichier brut introuvable : {RAW_CSV_PATH}"

# Chargement des données brutes
df_raw = pd.read_csv(RAW_CSV_PATH)

# Safety check : le DataFrame brut ne doit pas être vide
assert not df_raw.empty, "❌ Le DataFrame brut est vide. Vérifier le scraping / le chemin."

print("✅ Fichier brut chargé avec succès.")
print("Shape brut (lignes, colonnes) :", df_raw.shape)

display(df_raw.head())


On affiche les 5 premières lignes du dataframe.\
Le fichier brut contient `df_raw.shape[600]` annonces pour `df_raw.shape[13]` variables, dont :
- `url`
- `type_bien`
- `titre`
- `prix`
- `surface_habitable`
- `surface_terrain`
- `pieces`
- `ville`
- `cp`
- `dpe`
- `ges`
- `features`
- `description`

In [ ]:
# 1.1.2. Structure, types et valeurs manquantes (df_raw)

print("=== Informations générales sur le DataFrame brut ===\n")
df_raw.info()

print("\n=== Nombre de valeurs manquantes par colonne ===")
na_counts = df_raw.isna().sum()
display(na_counts)


Cette étape sert à :

1. **Contrôler la structure du fichier brut**
   - `df_raw.info()` vérifie les types de colonnes :
      - Cela permet de vérifier si les colonnes sont dans un format cohérent avec leur usage :
     - `prix`, `surface_habitable`, `surface_terrain`, `pieces` devraient être numériques ;
     - `ville`, `cp`, `type_bien`, `dpe`, `ges`, `features` sont des variables textuelles / catégorielles.

2. **Quantifier les valeurs manquantes**
   - `df_raw.isna().sum()` donne le nombre de `NaN` par colonne.
   - Les colonnes avec beaucoup de `NaN` (par exemple `surface_terrain`, `dpe`, `ges`, `features`) reflètent :
     - soit un **problème de scraping** (données non récupérées),
     - soit un **manque d’information dans les annonces** (par exemple, diagnostics DPE/GES non renseignés).
   - Ces résultats sont importants pour deux raisons :
     - orienter les choix de nettoyage (quelles colonnes peuvent être utilisées directement, lesquelles nécessitent un traitement particulier),
     - préparer un **audit de qualité des données** dans le rapport (taux de remplissage par variable).

En dehors de ces aspects quantitatifs, on s'interroge sur la qualité de la donnée, incluant son taux de complétude.\
En l'occurrence, les informations mandatory sont bien présentes, ce qui permet de construire une base CLEAN exploitable. Les catégories qui souffrent d'un fort taux de valeurs manquantes sont en réalité des informations complémentaires (surface du terrain, des bonus comme le parking, et les diagnostics qui peuvent être manquants lors de la publication de l'annonce), qui apportent une valeur ajoutée au bien. Ces valeurs seront interprétées en gardant en tête ces taux de complétude.

In [ ]:
# 1.1.3. Statistiques descriptives des colonnes numériques (df_raw)

# Colonnes numériques attendues d'après le scraping
num_cols_expected = ["prix", "surface_habitable", "surface_terrain", "pieces"]
num_cols = [c for c in num_cols_expected if c in df_raw.columns]

print("Colonnes numériques présentes :", num_cols)

display(df_raw[num_cols].describe())


Cette cellule affiche un premier résumé statistique des principales variables numériques à partir du fichier brut :
- `count` : nombre de valeurs non manquantes.
- `mean` (moyenne) : valeur moyenne observée.
- `std` (écart-type) : mesure de la dispersion autour de la moyenne.
- `min` : plus petite valeur observée.
- `25%`, `50%` (médiane), `75%` : quartiles, qui indiquent comment se répartissent les données.
- `max` : plus grande valeur observée.

L'objectif n’est pas encore de tirer des conclusions économiques, mais de faire un contrôle qualité :
- Elles incluent encore :
  - des **doublons**,
  - des **valeurs potentiellement aberrantes** (prix = 0, surfaces très faibles),
  - des **outliers** (biens très chers, grandes propriétés),
  - des erreurs possibles liées au scraping.
- L’objectif de cette `describe()` sur le brut est surtout :
  - de vérifier les **ordres de grandeur** (par exemple, vérifier qu’on est bien dans des prix cohérents pour des maisons dans le secteur bordelais),
  - de repérer la présence de valeurs extrêmes (min / max).

La véritable interprétation métier (prix typique, distribution des prix/m², etc.) viendra plus tard, **après le nettoyage**, sur le dataset CLEAN.

In [ ]:
# 1.1.4. Aperçu des variables catégorielles (df_raw)

cat_cols = [c for c in ["type_bien", "ville", "cp", "dpe", "ges"] if c in df_raw.columns]

for col in cat_cols:
    print(f"\n=== Top modalités pour {col} ===")
    display(df_raw[col].value_counts().head(10))
    

Cette cellule explore la distribution de quelques variables catégorielles importantes :
- `type_bien` : typologie du bien (maison, appartement, etc.). Dans ce projet, on se concentre sur les maisons, mais cette colonne permet de vérifier que le filtrage par type a bien été appliqué dans le scraping.
- `ville` : lieu principal de l’annonce. La distribution par ville permet de voir quelles communes du rayon R10 autour de Bordeaux sont le plus représentées.
- `cp` : code postal, utile pour des analyses plus fines par zone.
- `dpe`, `ges` : classes énergétiques (Diagnostic de Performance Energétique et Gaz à Effet de Serre). Ces variables peuvent contenir beaucoup de valeurs manquantes ; la distribution donne une première idée de la complétude des diagnostics.

Ce comptage (Top 10 des modalités) sert à :

- **identifier les catégories dominantes** (ex : certaines villes très présentes),
- repérer les catégories rares ou absentes,
- préparer des **analyses croisées** plus tard :
  - prix/m² par ville,
  - prix/m² selon la classe énergétique,
  - etc.

À ce stade, on ne tire pas encore de conclusion forte, on se contente de comprendre **comment le marché est représenté** dans le fichier brut.

In [ ]:
# 1.1.5. Diagnostic des doublons (df_raw)

# Doublons stricts : toutes colonnes identiques
nb_dup_strict = df_raw.duplicated().sum()
print(f"Nombre de lignes strictement dupliquées (toutes colonnes identiques) : {nb_dup_strict}")

# Doublons potentiels basés sur l'URL
if "url" in df_raw.columns:
    dup_url_mask = df_raw.duplicated(subset=["url"], keep=False)
    df_dup_url = df_raw[dup_url_mask].sort_values("url")
    print(f"\nNombre d'URLs apparaissant plusieurs fois : {df_dup_url['url'].nunique()}")
    print(f"Nombre total de lignes impliquées dans des doublons d'URL : {df_dup_url.shape[0]}")
    display(df_dup_url.head(10))
else:
    print("\n⚠ La colonne 'url' n'existe pas, impossible de diagnostiquer les doublons d'URL.")
    

**Remarque sur les doublons**

Ici, on veut en théorie supprimer uniquement :
- les **doublons stricts** (lignes identiques sur toutes les colonnes) ;
- les **doublons d’URL** (même annonce technique répétée).

Les résultats nuls sont expliqués par le fait que ces types de doublons ne concernent pas notre étude. En effet, toutes les annonces proviennent de la même plateforme Etreproprio.com à un seul instant donné. Par défaut, aucune url ne peut donc être en double.

On veut pouvoir nettoyer les artefacts techniques du scraping, mais cela **ne traitera pas** les cas plus complexes où **une même maison est publiée par plusieurs agences différentes** avec des URLs différentes (doublons inter-agences).  

Ces doublons "métier" peuvent légèrement biaiser certaines statistiques (par exemple les prix moyens par ville). Ils ne sont pas résolus dans ce projet, mais ils sont clairement identifiés comme une **limite** et une **piste d’amélioration** (déduplication basée sur le croisement de plusieurs critères : ville, code postal, surface, pièces, prix, etc.).

In [ ]:
# 1.1.6. Détection des doublons (df_raw)

# ---------------------------------------------------------
# SAFETY CHECK AVANCÉ : DÉTECTION DES DOUBLONS MULTI-AGENCES
# ---------------------------------------------------------

# On cherche les annonces qui ont strictement :
# - Même code postal
# - Même surface habitable
# - Même nombre de pièces
# - Un prix très proche (ex: écart < 5%)

cols_doublons = ['cp', 'surface_habitable', 'pieces']

# On compte les occurrences sur ces critères stricts
potential_dups = df_raw.duplicated(subset=cols_doublons, keep=False)

if potential_dups.any():
    print(f"⚠️ ATTENTION : {potential_dups.sum()} annonces semblent être des doublons multi-agences (même CP, surface, pièces).")
    
# Affichons un exemple pour vérifier visuellement
    sample_dup = df_raw[potential_dups].sort_values(by=cols_doublons).head(4)
    display(sample_dup[['titre', 'ville', 'prix', 'surface_habitable', 'pieces', 'url']])
    
    print("Conseil Analyse : Ces lignes peuvent biaiser légèrement la moyenne des prix.")
else:
    print("✅ Aucune suspicion forte de doublons multi-agences détectée sur critères stricts.")
    

### Synthèse : Détection des doublons multi-agences

1. **La mécanique (comment ça marche ?)**

- **L’empreinte “physique”** :  
  On définit un candidat doublon par le trio `Code Postal + Surface habitable + Nombre de pièces`. C’est une signature “physique” du bien, appliquée ici de façon **stricte** (égalité exacte sur ces trois colonnes, sans critère direct sur le prix).

- **Le radar (`keep=False`)** :  
  On utilise `duplicated(subset=..., keep=False)` pour marquer **toutes** les lignes appartenant à un groupe de lignes identiques sur ces trois caractéristiques (l’originale et les duplicatas). On obtient ainsi l’ensemble des annonces “suspectes”.

- **L’affichage pour audit** :  
  On trie ces lignes par `cp`, `surface_habitable` et `pieces` pour regrouper visuellement les “jumeaux” et comparer ensuite, à l’œil, leurs titres, prix et URLs.

2. **L’interprétation (analyse métier)**

- **Le phénomène visé** :  
  Cette méthode met en évidence des **candidats à des “mandats simples”** (multi-diffusion) : une même maison publiée simultanément par plusieurs agences (Orpi, IAD, Century21, etc.), avec des annonces très proches sur le plan physique.

- **Insight prix (lecture qualitative)** :  
  En inspectant ces groupes à la main, on constate souvent que le “prix de marché” n’est pas un point mais une fourchette : d’une agence à l’autre, on observe des écarts de prix modestes (souvent de l’ordre de 1 à 5 %), qui reflètent notamment des honoraires ou stratégies de pricing différents.

- **Les biais induits** :
  - **Volume** : l’offre semble plus vaste qu’elle ne l’est réellement, car un même bien apparaît plusieurs fois, sous plusieurs mandats.  
  - **Moyenne** : un bien cher répliqué 2 ou 3 fois peut tirer artificiellement vers le haut la moyenne des prix d’un quartier ou d’une ville.

3. **La limite technique (safety check)**

- **Détection stricte** :  
  L’algorithme cherche une égalité parfaite sur `cp`, `surface_habitable` et `pieces`. Il ne tient pas compte directement du prix et ne voit pas, par exemple, les cas où la surface diffère légèrement (100 m² vs 100,5 m²) ou le nombre de pièces est sujet à interprétation (mezzanine comptée ou non).

- **Le piège assumé** :  
  En étant strict, on **manque certains vrais doublons** (multi-agences avec légères variations d’arrondi ou de saisie), mais on limite le risque inverse de considérer comme doublons deux maisons réellement différentes mais très ressemblantes. C’est un compromis volontaire : ce bloc sert de **safety check** exploratoire, pas de base automatique de dédoublonnage métier.

### Analyse critique de la détection des doublons multi-agences

Notre algorithme de détection (basé sur le triptyque Code Postal / Surface / Pièces) a identifié 71 annonces suspectes, soit environ 12% du dataset.

Cependant, une analyse qualitative des résultats montre la limite de cette approche sur un marché dense comme Bordeaux. Nous observons deux cas de figure :
- Les doublons avérés : des biens avec des caractéristiques identiques et un prix très proche (écart < 5%), confirmant la multi-diffusion inter-agences.
- Les faux positifs : des biens partageant les mêmes métriques (ex: 155m² / 6 pièces) mais avec des écarts de prix massifs (> 20%). Il s'agit ici de biens distincts, illustrant l'hétérogénéité du marché bordelais où, à surface égale, l'état du bien ou le micro-quartier impactent fortement le prix.

Décision méthodologique :
Face au risque de supprimer des biens distincts (faux positifs), nous avons choisi de conserver ces lignes dans le jeu de données final, en acceptant un léger bruit statistique plutôt qu'une perte d'information.

## Bilan du diagnostic des données brutes (df_raw)

Avant nettoyage, le fichier brut présente les caractéristiques suivantes :

- **Volume et structure**  
  - 600 lignes d'annonces, pour 13 colonnes de variables (url, prix, surface, pièces, localisation, diagnostics, etc.).
  - Les types sont globalement cohérents : prix et surfaces sont numériques après conversion, les champs de localisation et diagnostics sont textuels.

- **Qualité et complétude**  
  - Les variables clés pour la problématique (`prix`, `surface_habitable`, `ville`, `cp`) sont bien renseignées.
  - Certaines colonnes présentent un **taux de valeurs manquantes non négligeable** : `surface_terrain`, `features`, `dpe`, `ges`.  
    Cela reflète des annonces plus ou moins complètes plutôt qu’un problème de scraping.

- **Doublons**  
  - Pas de duplication technique massive (doublons stricts ou d’URL).  
  - En revanche, des **candidats doublons “multi-agences”** existent (même CP, surface, pièces), ce qui peut légèrement gonfler le volume et biaiser certaines moyennes.  
    Ces doublons métier sont **identifiés mais conservés**, et seront éventuellement discutés en limite.

- **Géolocalisation**  
  Le fichier brut ne contient pas de coordonnées GPS (`lat`, `lon`) : le site scrapé ne mentionne pas cette information fine, confirmé par une 1ère version du scraper qui avait la fonctionnalité (et qui n'a donc pas été conservée) de récupérer ces données mais qui se sont avérées être vides.
  En revanche, les colonnes `ville` et `cp` sont bien renseignées et serviront de base à une étape ultérieure de **géocodage** (via Nominatim / OpenStreetMap) pour le fichier CLEAN construit.
  Cette géolocalisation dérivée ne fait donc pas partie du diagnostic du brut, mais sera utilisée ensuite pour préparer la future carte (folium) et le futur dashboard.

- **Implications pour la suite**  
  - Les données brutes sont **suffisamment structurées** pour construire un fichier CLEAN exploitable.
  - Le nettoyage va surtout porter sur :
    - la mise au bon format des variables numériques,
    - un filtrage simple des cas aberrants (prix ≤ 0, surfaces trop faibles),
    - la construction de la variable clé `prix_m2`,
    - la préparation d’un sous-échantillon “cœur de marché” pour certaines analyses
    - et le géocodage par données cp + ville pour la visualisation

Ce bilan sert de point de départ : à partir d’ici, on bascule du **diagnostic de qualité de données** vers le **nettoyage et la structuration**.

## 1.2. **Construction d’une base propre (`df_clean`, `df_final`)**

Note sur le workflow du fichier de données :\
Ici on passe à un dataFrame de travail (df), puis df_clean, df_clean_iqr en complément, et enfin df_final qu’on exporte.

In [ ]:
# 1.2.1. Création d’une copie de travail df

# On ne touche plus à df_raw : c'est l'archive brute.
df = df_raw.copy()

print("Shape de df (copie de travail) :", df.shape)


**Pourquoi utiliser `df = df_raw.copy()` ?**

Par principe, nous ne modifions jamais le fichier original en place : `df_raw` contient donc les données **telles qu’elles sortent du scraping**. 
Toutes les opérations de nettoyage sont réalisées sur une copie (df = df_raw.copy()), puis sur des versions dérivées.

Le fichier brut issu du scraping (df_raw) est considéré comme une trace d’audit de la collecte, ce qui permet de :
- **préserver le brut** en référence (contrôle qualité, possibilité de revenir en arrière) ;
- pouvoir comparer “avant / après” nettoyage ;
- éviter de modifier par erreur la source originale ;
- rendre le pipeline plus lisible : `df_raw` = source, `df` = données de travail, `df_explo` = données prêtes pour l’analyse.

C’est une bonne pratique en data.

In [ ]:
# 1.2.2. Suppression des doublons (stricts + URL)
"""
Consigne du sujet : supprimer les doublons à l'étape de nettoyage.
-- on enlève les doublons stricts (lignes identiques)
-- on enlève les doublons d'URL (on garde la première annonce par URL)."""

# Suppression des doublons stricts (toutes colonnes identiques)
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Doublons stricts supprimés : {before - after} lignes.")
print("Shape après suppression des doublons stricts :", df.shape)

# Suppression des doublons basés sur l'URL
if "url" in df.columns:
    before_url = df.shape[0]
    df = df.drop_duplicates(subset=["url"], keep="first")
    after_url = df.shape[0]
    print(f"Doublons basés sur l'URL supprimés : {before_url - after_url} lignes.")
    print("Shape après suppression des doublons d'URL :", df.shape)
else:
    print("⚠ Pas de colonne 'url', pas de suppression de doublons d'URL.")
    

### Suppression des doublons techniques

Pour respecter les consignes du projet, cette étape applique une stratégie de nettoyage **technique** :

1. **Doublons stricts**  
   On supprime d’abord les lignes strictement identiques sur toutes les colonnes (`df.drop_duplicates()`),\
   ce qui élimine les éventuels artefacts de scraping (rechargement de page, relance de script, etc.).

2. **Doublons d’URL**  
   On supprime ensuite les doublons basés sur la colonne `url` `drop_duplicates(subset=["url"], keep="first")`),\
   en considérant qu’une même URL correspond à une même annonce technique.

Dans ce projet, comme vu plus haut, le dataset n'est pas réellement concerné (contrôlé par les messages `Doublons stricts supprimés : ...` et `Doublons basés sur l'URL supprimés : ...`), ce qui confirme que le scraping n’a pas généré de duplication massive.

Les **doublons métier** (la même maison en vente publiée par plusieurs agences, avec des URLs différentes) ont été détectés et discutés séparément dans la partie "doublons multi-agences",\
mais **ne sont pas supprimés automatiquement** pour éviter d’enlever par erreur des biens distincts.

In [ ]:
# 1.2.3. Normalisation des types (numériques + cp)
"""
On s'assure que les colonnes numériques sont bien au bon format, et que cp est une chaîne (pour conserver les zéros initiaux si on généralise à la France).
Le code postal doit absolument rester une chaine de caractères pour pouvoir afficher les 5 digits commençant par le chiffre 0."""

# Colonnes que l'on souhaite en type numérique
num_cols_to_numeric = ["prix", "surface_habitable", "surface_terrain", "pieces"]

print("=== Log des conversions numériques (NaN avant / après) ===")
for col in num_cols_to_numeric:
    if col in df.columns:
        na_before = df[col].isna().sum()
        df[col] = pd.to_numeric(df[col], errors="coerce")
        na_after = df[col].isna().sum()
        print(f"{col} : NaN avant = {na_before}, après conversion = {na_after}")
    else:
        print(f"{col} : colonne absente du DataFrame")

# Code postal en chaîne de caractères
if "cp" in df.columns:
    df["cp"] = df["cp"].astype("string")

print("\n=== Types après normalisation ===")
df.info()


### Log des conversions numériques et valeurs manquantes

Lors du passage en numérique (`pd.to_numeric`), on affiche le **nombre de NaN avant et après** conversion pour :
- `prix`
- `surface_habitable`
- `surface_terrain`
- `pieces`

Objectif :
- vérifier si la conversion a **créé des NaN supplémentaires** (par exemple lorsque des chaînes non numériques sont rencontrées) ;
- documenter cette étape dans le pipeline de nettoyage.

Ce log permet de montrer que :
- la conversion en type numérique n’a pas “magiquement corrigé” des valeurs aberrantes,
- au contraire, elle **rend explicites** les valeurs non interprétables (qui deviennent des NaN et seront ensuite traitées par le filtrage basique).

In [ ]:
# 1.2.4. Identification des annonces incomplètes (prix ou surface manquants / inutilisables)

conditions_incomplets = (
    df["prix"].isna() |
    df["surface_habitable"].isna() |
    (df["prix"] <= 0) |
    (df["surface_habitable"] <= 10)
)

df_incomplets = df[conditions_incomplets].copy()

print(f"Nombre d'annonces incomplètes (prix ou surface manquants / invalides) : {len(df_incomplets)}")
display(df_incomplets.head())


### Focus sur les annonces incomplètes (`df_incomplets`)

Avant de filtrer vers `df_clean`, on isole dans `df_incomplets` les annonces :
- sans `prix` ou avec `prix ≤ 0`,
- sans `surface_habitable` ou avec `surface_habitable ≤ 10 m²`.

Ces annonces sont **inexploitables** pour le calcul du prix au m², mais elles sont intéressantes pour :
- documenter la **qualité du fichier brut** (annonces partiellement renseignées),
- quantifier la part du marché qui ne peut pas être utilisée dans les analyses principales.

Elles seront écartées de `df_clean`, mais `df_incomplets` reste disponible comme **trace d’audit** dans le notebook.

In [ ]:
# 1.2.5. Création de la variable clé prix_m2 (brut)

# Vérification de la présence des colonnes nécessaires
assert "prix" in df.columns and "surface_habitable" in df.columns, \
    "Colonnes 'prix' ou 'surface_habitable' manquantes."

# Calcul initial du prix au m² (sans filtrage)
df["prix_m2"] = df["prix"] / df["surface_habitable"]

print("Aperçu de prix, surface_habitable, prix_m2 :")
display(df[["prix", "surface_habitable", "prix_m2"]].head(10))

# On arrondit les statistiques à 0 décimale pour un affichage lisible
stats_pm2 = df["prix_m2"].describe().round(0)
display(stats_pm2)


Il était important de créer cette variable, afin d'avoir une donnée clé en matière de marché immobilier.

Dans cette étape, on crée une variable centrale pour notre problématique :
> `prix_m2 = prix / surface_habitable`

Cette variable permet de comparer des biens de tailles différentes sur une base commune (le prix au mètre carré), ce qui est précisément au cœur de la question :  
*« Comment le prix au mètre carré varie-t-il selon la localisation, la surface et les caractéristiques du bien ? »*

La `describe()` sur `prix_m2` avant filtrage basique donne une idée :
- de la **moyenne** et de la **médiane** :
  - si la moyenne est nettement supérieure à la médiane, cela signifie que quelques biens très chers tirent la moyenne vers le haut (distribution asymétrique avec une “queue” haute).
- des **quartiles** (25 %, 50 %, 75 %) :
  - ils montrent dans quelle fourchette se situe la majorité des biens.
- du **min** et du **max** :
  - les valeurs extrêmes peuvent correspondre à des problèmes de données (ex : surface mal scrapée) ou à des biens très atypiques (ville très cotée, villa de luxe).

À ce stade, `prix_m2` est encore calculé sur un jeu qui peut contenir :
- des prix nuls ou incohérents,
- des surfaces trop petites,
- des NaN.

L’objectif ici est surtout de **vérifier que la formule est correcte** et que les ordres de grandeur sont raisonnables, avant d’appliquer un premier filtrage qualitatif.

In [ ]:
# 1.2.6. Filtrage basique → df_clean

# Filtrage
"""
On retire les cas clairement inutilisables :
-- prix ≤ 0 ou NaN,
-- surface_habitable trop faible ou NaN."""

df_clean = df.copy()

mask_valid = (
    df_clean["prix"].notna() &
    df_clean["surface_habitable"].notna() &
    (df_clean["prix"] > 0) &
    (df_clean["surface_habitable"] > 10)
)

before = df_clean.shape[0]
df_clean = df_clean[mask_valid].copy()
after = df_clean.shape[0]

print(f"Filtrage basique (prix > 0, surface_habitable > 10, non-NaN) : {before} -> {after} lignes.")

# Recalcul de prix_m2 après ce premier nettoyage
df_clean["prix_m2"] = df_clean["prix"] / df_clean["surface_habitable"]

print("\nStatistiques sur prix_m2 après filtrage basique :")
display(df_clean["prix_m2"].describe())


On applique ici un premier filtrage de qualité pour construire `df_clean` :

On conserve uniquement les annonces pour lesquelles :
  - le `prix` est renseigné et strictement positif ;
  - la `surface_habitable` est renseignée et supérieure à 10 m²
    (seuil minimal pour exclure les cas aberrants ou des erreurs de saisie).

Concrètement, on passe de `before` à `after` lignes, comme indiqué dans le log `Filtrage basique (prix > 0, surface_habitable > 10, non-NaN) : ...`.

Ce filtrage permet d’éliminer :
- des biens manifestement incorrects (prix = 0, surface très faible) ;
- des lignes inexploitées pour le calcul du prix au m² (prix ou surface manquants).

On recalcule ensuite `prix_m2` sur `df_clean` et on examine à nouveau les statistiques (`describe()`), qui sont désormais basées sur un **jeu de données plus fiable**.  
C’est ce `df_clean` qui sert de base à la construction du fichier CLEAN officiel.

In [ ]:
# 1.2.7. Option d'analyse : sous-échantillon filtré 5–95 % sur prix_m2 (non utilisé pour le fichier CLEAN officiel)

# Calcul des seuils de prix_m2 aux percentiles 5% et 95%
q05 = df_clean["prix_m2"].quantile(0.05)
q95 = df_clean["prix_m2"].quantile(0.95)

print(f"Seuil 5% prix_m2 : {q05:.2f}")
print(f"Seuil 95% prix_m2 : {q95:.2f}")

mask_iqr = (df_clean["prix_m2"] >= q05) & (df_clean["prix_m2"] <= q95)

before_iqr = df_clean.shape[0]
df_clean_iqr = df_clean[mask_iqr].copy()
after_iqr = df_clean_iqr.shape[0]

print(f"\nFiltrage par percentiles 5%-95% sur prix_m2 : {before_iqr} -> {after_iqr} lignes.")

print("\nStatistiques sur prix_m2 avant filtrage des extrêmes :")
display(df_clean["prix_m2"].describe())
print("\nStatistiques sur prix_m2 après filtrage des extrêmes :")
display(df_clean_iqr["prix_m2"].describe())


Sous-échantillon 5–95 % : cœur de marché (`df_clean_iqr`)

Pour certaines analyses globales (graphes, moyennes par segment), on travaille sur un sous-échantillon `df_clean_iqr` qui ne garde que les biens situés entre les **5ᵉ et 95ᵉ percentiles** de `prix_m2`:
- `q05` : seuil bas de prix/m² (5 % des biens ont un prix/m² inférieur)  
- `q95` : seuil haut de prix/m² (5 % des biens ont un prix/m² supérieur)

On obtient ainsi un **“cœur de marché”** où :
- les valeurs minimales et maximales de `prix_m2` sont moins extrêmes ;
- l’écart-type est réduit ;
- les graphiques sont plus lisibles et les moyennes moins sensibles aux biens
  très atypiques (grandes propriétés, localisations ultra-premium, etc.).

Important : ce filtrage n’est **pas** utilisé pour construire le fichier CLEAN officiel : les outliers restent conservés dans `df_clean` pour pouvoir être analysés séparément
(biens de prestige, quartiers très chers, cas extrêmes).

## Bilan intermédiaire du nettoyage

À ce stade, on dispose de deux bases de travail :
- `df_clean` : jeu de données **nettoyé mais complet**, incluant l’ensemble des annonces utilisables, **y compris les valeurs extrêmes** de `prix_m2`. C’est la base de référence pour le fichier CLEAN et pour l’analyse des outliers.
- `df_clean_iqr` : sous-échantillon limité au **cœur de marché**, défini entre les 5ᵉ et 95ᵉ percentiles de `prix_m2`. Il est utilisé uniquement pour certaines analyses globales (moyennes, comparaisons par segment, graphes plus lisibles), afin de réduire l’influence des biens très atypiques.

La suite du projet s’appuiera donc :
- sur `df_clean` lorsque l’objectif est de décrire **le marché dans son ensemble**, y compris les biens d’exception ;
- sur `df_clean_iqr` lorsqu’on veut mettre en avant des tendances “moyennes” plus stables, moins sensibles aux outliers.

In [ ]:
# Construction d’une base propre (`df\_clean`, `df\_final`)

# Pour la suite, on décide que c’est df_clean (filtre basique, outliers conservés) qui devient la base de référence pour le fichier CLEAN et les analyses.

# On part de df_clean (filtrage basique, outliers conservés)

df_for_export = df_clean.copy()

print("Shape de df_for_export (base pour le CLEAN) :", df_for_export.shape)
display(df_for_export.head())


## 1.3. Enrichissement : géolocalisation (Latitude/Longitude)

In [ ]:
# 1.3.1. Extraction des couples (ville, cp) uniques à géocoder

# Safety : vérifier que les colonnes ville et cp existent dans df_for_export
assert "ville" in df_for_export.columns and "cp" in df_for_export.columns, \
    "Colonnes 'ville' et/ou 'cp' manquantes dans df_for_export."

print(f"Nombre de lignes dans df_for_export : {len(df_for_export)}")

df_localites = (
    df_for_export[["ville", "cp"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Nombre de couples (ville, cp) uniques à géocoder : {len(df_localites)}")
display(df_localites.head())


In [ ]:
# 1.3.2. Géocodage (ville, cp) -> (lat, lon) avec Nominatim

# Initialisation du geocoder Nominatim
geolocator = Nominatim(user_agent="du_sda_immo_bordeaux")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # 1 requête / seconde minimum

def build_query(row):
    """
    Construit une chaîne de requête pour Nominatim à partir de (cp, ville)
    Exemple : "33000 Bordeaux, France"
    """
    cp_val = row["cp"]
    ville_val = row["ville"]

    if pd.isna(cp_val) or str(cp_val).strip() == "":
        return f"{ville_val}, France"
    else:
        return f"{cp_val} {ville_val}, France"

coords = []

for idx, row in df_localites.iterrows():
    query = build_query(row)
    try:
        location = geocode(query)
        if location:
            lat = location.latitude
            lon = location.longitude
        else:
            lat = None
            lon = None
    except Exception as e:
        print(f"⚠ Erreur de géocodage pour '{query}' : {e}")
        lat = None
        lon = None

    coords.append({
        "ville": row["ville"],
        "cp": row["cp"],
        "lat": lat,
        "lon": lon
    })

    # Optionnel : afficher de temps en temps la progression
    if (idx + 1) % 20 == 0:
        print(f"{idx + 1} / {len(df_localites)} localités géocodées...")

df_coords = pd.DataFrame(coords)
print("\nAperçu des géolocalisations obtenues :")
display(df_coords.head())


In [ ]:
# 1.3.3. Jointure des coordonnées sur df_for_export, construction de df_final et sauvegarde du CLEAN

# Jointure lat / lon sur df_for_export
df_for_export = df_for_export.merge(df_coords, on=["ville", "cp"], how="left")

print("\nVérification des nouvelles colonnes lat / lon dans df_for_export :")
display(df_for_export[["ville", "cp", "lat", "lon"]].head())

# Safety check : proportion de lignes sans géolocalisation
nb_missing_geo = df_for_export["lat"].isna().sum()
print(f"\nNombre de lignes sans géolocalisation : {nb_missing_geo} "
      f"({nb_missing_geo / len(df_for_export) * 100:.1f} %)")

# Colonnes utiles pour le fichier CLEAN (analyse + dashboard + cartographie)
colonnes_utiles = [
    "url",
    "type_bien",
    "titre",
    "prix",
    "surface_habitable",
    "surface_terrain",
    "pieces",
    "ville",
    "cp",
    "dpe",
    "ges",
    "features",
    "description",
    "prix_m2",
    "lat",
    "lon",
]

colonnes_existantes = [c for c in colonnes_utiles if c in df_for_export.columns]
df_final = df_for_export[colonnes_existantes].copy()

print("\nColonnes retenues pour le fichier CLEAN :", colonnes_existantes)
print("Shape du DataFrame final (CLEAN) :", df_final.shape)
display(df_final.head())

assert not df_final.empty, "❌ Le DataFrame final est vide : filtrage trop sévère."

# Sauvegarde du fichier CLEAN
df_final.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"\n✅ Fichier CLEAN sauvegardé sous : {CLEAN_CSV_PATH}")


# Le fichier CLEAN a bien été généré et exporté `data_bordeaux_R10_maisons_CLEAN.csv`.

## Bilan de la phase de nettoyage et géolocalisation

À l’issue de cette étape, on dispose désormais d’une base **CLEAN** prête pour l’analyse et la visualisation :

- `df_clean` :  
  - données nettoyées à partir de `df_raw` (suppression des doublons techniques, typage numérique, filtrage basique des cas aberrants, création de `prix_m2`) ;  
  - conservation des valeurs extrêmes (outliers) pour pouvoir analyser les biens d’exception.

- `df_clean_iqr` :  
  - sous-échantillon de **cœur de marché**, limité au [5ᵉ ; 95ᵉ] percentile de `prix_m2` ;  
  - utilisé uniquement pour certaines analyses globales (moyennes, comparaisons par segment), afin de limiter l’influence des biens très atypiques.

- **Géolocalisation** :  
  - les couples `(ville, cp)` uniques ont été géocodés via Nominatim (OpenStreetMap) ;  
  - les coordonnées `lat` et `lon` ont été jointes à la base via `df_for_export.merge(df_coords, ...)` ;  
  - le pourcentage de lignes non géocodées est affiché dans le log et reste acceptable pour une utilisation cartographique.

- `df_final` :  
  - DataFrame final contenant les colonnes utiles pour la suite du projet :  
    `prix`, `surface_habitable`, `pieces`, `ville`, `cp`, `prix_m2`, `dpe`, `ges`, `features`, ainsi que `lat` et `lon` ;  
  - exporté sous la forme du fichier CSV CLEAN :  
    **`DATA/data_bordeaux_R10_maisons_CLEAN.csv`**.

La prochaine section du notebook recharge ce fichier CLEAN dans `df_explo` et se concentre sur l’**exploration statistique** (describe, histogrammes, boxplots, corrélations, analyses croisées).\
Le projet ne va pas jusqu’à une géolocalisation fine à l’adresse ou au niveau IRIS, par choix de complexité et parce que le site source ne fournit pas ces informations de façon fiable.

## 2. **Exploration & analyses**

In [ ]:
# Chargement du CLEAN pour exploration (df_explo)

assert os.path.exists(CLEAN_CSV_PATH), f"❌ Fichier CLEAN introuvable : {CLEAN_CSV_PATH}"

df_explo = pd.read_csv(CLEAN_CSV_PATH)

print("✅ Fichier CLEAN chargé pour exploration.")
print("Shape df_explo :", df_explo.shape)
df_explo.info()

display(df_explo.head())

# Nous avons choisi de conserver les outliers dans df_explo. Tout ce qui suit (describe, histogrammes, boxplots, corrélations) se fait donc sur le marché réel avec ses valeurs extrêmes, afin de pouvoir analyser les biens très haut de gamme, les quartiers très cotés, etc.

In [ ]:
# 2.1. Stats descriptives sur prix, surface_habitable, prix_m2

# Stats effectuées sur les données nettoyées (`df_explo`, issues du fichier CLEAN)

num_cols_explo = ["prix", "surface_habitable", "surface_terrain", "pieces", "prix_m2"]
num_cols_explo = [c for c in num_cols_explo if c in df_explo.columns]

print("Colonnes numériques analysées :", num_cols_explo)
display(df_explo[num_cols_explo].describe())


Cette cellule présente les statistiques descriptives des variables numériques principales, mais cette fois **sur les données nettoyées** (`df_explo`, issues du fichier CLEAN) :
- `prix` : prix absolu du bien.
- `surface_habitable` : taille du logement.
- `surface_terrain` : taille du terrain (quand disponible).
- `pieces` : nombre de pièces.
- `prix_m2` : prix au mètre carré, variable centrale du projet.

Les indicateurs commentés en priorité sont :
- **Moyenne vs médiane de `prix_m2` :**
  - si la moyenne est > médiane : quelques biens très chers tirent la moyenne vers le haut,
  - si elles sont proches : marché plus homogène.
- **Quartiles de `prix_m2` (25 %, 50 %, 75 %) :**
  - ils indiquent “dans quelle fourchette” se situe la majorité des biens.
- **Min / max :**
  - permettent de repérer les biens les moins chers et les plus chers dans le secteur,
  - à croiser ensuite avec la localisation et les caractéristiques (piscine, terrain, etc.).

Cette `describe()` sur le CLEAN est la première base solide pour répondre à la question :  
*« Quel est le niveau de prix au mètre carré typique dans le secteur bordelais étudié ? »*

Les mêmes indicateurs sont aussi disponibles pour prix, surface_habitable, etc., mais on se concentre ici sur prix_m2 qui est la variable centrale de l’analyse.

In [ ]:
# 2.2. Calculs BONUS

# 2.2.1. Bonus Moyenne et médiane du prix au m²
mean_pm2 = df_explo["prix_m2"].mean()
median_pm2 = df_explo["prix_m2"].median()

print(f"Moyenne prix/m² : {mean_pm2:.0f} € / m²")
print(f"Médiane prix/m² : {median_pm2:.0f} € / m²")

# Rapport entre moyenne et médiane
ratio_mean_median = mean_pm2 / median_pm2
ecart_relatif = (mean_pm2 - median_pm2) / median_pm2 * 100

print(f"Rapport moyenne / médiane : {ratio_mean_median:.3f}")
print(f"Écart relatif moyenne vs médiane : {ecart_relatif:.1f} %")

# Phrase d'interprétation automatique (optionnel)
if ecart_relatif > 0:
    tendance = "supérieure à"
    interpretation = "ce qui suggère que quelques biens très chers tirent la moyenne vers le haut."
elif ecart_relatif < 0:
    tendance = "inférieure à"
    interpretation = "ce qui est peu fréquent dans ce type de données et peut refléter quelques biens très peu chers."
else:
    tendance = "égale à"
    interpretation = "ce qui indique une distribution très symétrique des prix au m²."

print(
    f"\nInterprétation : le prix/m² médian est d’environ {median_pm2:.0f} € / m², "
    f"avec une moyenne {tendance} la médiane ({mean_pm2:.0f} € / m²), "
    f"soit un écart relatif d’environ {ecart_relatif:.1f} %. "
    f"Globalement, {interpretation}"
)


Cette cellule calcule deux indicateurs clés pour le prix au m² (`prix_m2`) sur le jeu de données nettoyé :
- la **moyenne** : somme de tous les prix/m² divisée par le nombre de biens ;
- la **médiane** : valeur qui partage l’échantillon en deux (50 % des biens ont un prix/m² inférieur, 50 % supérieur).

Comparaison informative :
- si la moyenne est nettement plus élevée que la médiane :
  - la distribution est **asymétrique**, tirée par quelques biens très chers,
  - la médiane est souvent plus représentative du prix “typique” du marché.
- si la moyenne et la médiane sont proches :
  - la distribution est plus symétrique,
  - les outliers ont moins d’impact.

Dans notre cas, la moyenne du prix au m² ressort légèrement **supérieure** à la médiane, ce qui indique la présence de quelques biens très chers qui tirent la moyenne vers le haut. La médiane apparaît donc comme un indicateur plus représentatif du niveau de prix “typique” du marché que la moyenne arithmétique.

In [ ]:
# 2.2.2. Histogramme des prix

plt.figure(figsize=(8, 4))

# On enlève les valeurs manquantes pour éviter les warnings et les artefacts
prix_valides = df_explo["prix"].dropna()

plt.hist(prix_valides, bins=30)

plt.xlabel("Prix (€)")
plt.ylabel("Fréquence")
plt.title("Distribution des prix des maisons (Bordeaux R10 - données nettoyées)")

# Grille légère sur l'axe des y pour faciliter la lecture
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Évite que les labels soient coupés
plt.tight_layout()

plt.show()


Cet histogramme représente la **distribution des prix** des maisons (en euros), sur le jeu de données nettoyé :
- l’axe horizontal (x) : les tranches de prix (par exemple 200 000 €, 300 000 €, etc.) ;
- l’axe vertical (y) : le nombre de biens dans chaque tranche.

Ce graphique permet de voir :
- dans quelle zone de prix se concentre la majorité des biens (la “bosse” de l’histogramme),
- s’il existe une queue de distribution importante vers les prix élevés (biens haut de gamme),
- si on observe plusieurs “pics” qui pourraient correspondre à différents segments (petites maisons, grandes propriétés, etc.).

Interprétation :
- *“La plupart des maisons se situent dans une fourchette approximative de X à Y €.”*
- *“Quelques biens se détachent nettement au-dessus, ce qui confirme l’existence d’un segment très haut de gamme, cohérent avec le dynamisme de Bordeaux et de certains quartiers cotés.”*

In [ ]:
# 2.2.3. Histogramme des surfaces habitables

plt.figure(figsize=(8, 4))

# On enlève les valeurs manquantes pour éviter les warnings
surfaces_valides = df_explo["surface_habitable"].dropna()

plt.hist(surfaces_valides, bins=30)

plt.xlabel("Surface habitable (m²)")
plt.ylabel("Fréquence")
plt.title("Distribution des surfaces habitables (maisons, Bordeaux R10 – données nettoyées)")

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


Cet histogramme décrit la distribution de la **surface habitable** des maisons :
- axe horizontal : surface en m² ;
- axe vertical : nombre de biens dans chaque tranche de surface.

Ce graphique permet notamment de voir :
- si la majorité des maisons se concentre sur des surfaces “standard” (par exemple entre 80 et 140 m²) ;
- s’il existe un nombre significatif de très petites ou très grandes surfaces (petites échoppes, grandes propriétés, etc.).

En pratique, on peut commenter :
- la zone centrale (fourchette de surface la plus fréquente) ;
- la présence de quelques biens avec des surfaces très élevées, souvent corrélées aux niveaux de prix les plus élevés ;
- le lien possible avec la localisation (grandes surfaces plus fréquentes en périphérie qu’en centre urbain).

In [ ]:
# 2.2.4. Histogramme du prix/m²

plt.figure(figsize=(8, 4))

# Pour le tracé, on ignore seulement les NaN (ça ne modifie pas df_explo)
prix_m2_valides = df_explo["prix_m2"].dropna()

plt.hist(prix_m2_valides, bins=30)

plt.xlabel("Prix au m² (€ / m²)")
plt.ylabel("Fréquence")
plt.title("Distribution du prix au m² (maisons, Bordeaux R10 – données nettoyées)")

# Petite grille sur l'axe des y pour faciliter la lecture
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


Cet histogramme est directement lié à la problématique du projet. Il montre comment se répartit le **prix au mètre carré** dans le secteur :
- axe horizontal : prix/m² (en €/m²),
- axe vertical : nombre de biens dans chaque tranche de prix/m².

Ce graphique permet de :
- identifier la **zone de prix/m² la plus fréquente**,
- voir si la distribution est :
  - resserrée (marché homogène),
  - ou très étalée (fortes disparités entre quartiers ou segments de biens),
- repérer visuellement les biens dont le prix/m² est très supérieur au reste.

Interprétation possible :
- *“Le gros du marché se situe entre X et Y €/m², avec quelques biens au-delà de Z €/m² qui semblent correspondre à des biens d’exception (localisation premium, grandes surfaces, piscine, etc.).”*

In [ ]:
# 2.2.5. Boxplot du prix/m²

plt.figure(figsize=(7, 3))

# Boxplot horizontal, en utilisant toujours df_explo
df_explo.boxplot(column="prix_m2", vert=False)

plt.xlabel("Prix au m² (€ / m²)")
plt.title("Boxplot du prix au m² (maisons, Bordeaux R10 – données nettoyées)")
plt.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


Le boxplot est une représentation synthétique de la distribution de `prix_m2` :
- la **ligne au centre** de la boîte : la médiane (50 % des biens ont un prix/m² plus bas, 50 % plus haut) ;
- le bas de la boîte : 1er quartile (25 % des biens ont un prix/m² plus bas) ;
- le haut de la boîte : 3e quartile (75 % des biens ont un prix/m² plus bas) ;
- les “moustaches” : elles s’étendent généralement jusqu’à 1,5×IQR (intervalle interquartile) en dessous du 1er quartile et au-dessus du 3ᵉ quartile ; au-delà, les valeurs sont représentées comme des points isolés et interprétées comme des outliers ;
- les points isolés au-dessus (s’il y en a) : ce sont les **outliers** en prix/m².

Ce graphique est utile pour :
- quantifier rapidement la **dispersion** des prix/m²,
- repérer visuellement l’**écart** entre la médiane et les valeurs extrêmes,
- illustrer l’existence de biens très au-dessus du marché “typique”.

Interprétation type à ajouter :
- *“La médiane du prix/m² se situe autour de X €/m², ce qui correspond au niveau ‘typique’ du marché. On observe toutefois des valeurs nettement supérieures, témoignant de biens haut de gamme dans certains secteurs.”*

### Bilan intermédiaire sur la distribution des prix et des surfaces

Les histogrammes et le boxplot montrent un marché concentré autour d’un niveau de prix au m² “typique” (proche de la médiane calculée plus haut), avec une queue haute de distribution liée à quelques biens de prestige. Les surfaces habitables se situent majoritairement dans une fourchette “standard” (maisons familiales), mais on observe aussi des grandes propriétés qui expliquent une partie des valeurs élevées de prix total.

Ces constats motivent l’utilisation d’un sous-échantillon de **cœur de marché** (df_clean_iqr, entre les 5ᵉ et 95ᵉ percentiles de prix/m²) pour certaines analyses croisées, tout en conservant les outliers dans df_clean pour analyser les biens d’exception.

In [ ]:
# 2.3. Analyses poussées

# 2.3.1. Identification des outliers en prix/m² : top 10
N = 10

colonnes_outliers = [
    "ville",
    "cp",
    "prix",
    "surface_habitable",
    "surface_terrain",
    "prix_m2",
    "dpe",
    "ges",
    "features",
    "url"
]

colonnes_outliers = [c for c in colonnes_outliers if c in df_clean.columns]

top_outliers = (
    df_clean
    .dropna(subset=["prix_m2"])
    .sort_values("prix_m2", ascending=False)
    .head(N)[colonnes_outliers]
)

print(f"Top {N} biens les plus chers en prix/m² (df_clean) :")
display(top_outliers)


Pour mieux comprendre les **valeurs extrêmes** de `prix_m2`, on extrait le **Top N** des biens les plus chers au m² (top_outliers), en regardant :
- la ville / le code postal,
- le prix et la surface habitable,
- la surface de terrain,
- le DPE / GES,
- les principales `features` (piscine, vue, etc.).

Cet examen qualitatif permet de vérifier que les valeurs très élevées correspondent bien à :
- des localisations **premium**,
- des maisons de prestige,
- ou des biens avec des caractéristiques très différenciantes.

Ces outliers sont **volontairement conservés** dans `df_clean`, car ils font partie du marché réel, mais ils sont exclus du sous-échantillon `df_clean_iqr` utilisé pour analyser le “cœur de marché”.

In [ ]:
# 2.3.2. Comparaison du prix/m² selon la présence (ou non) d'un DPE / GES renseigné

def compare_prix_m2_diag(df, col_diag, label):
    assert col_diag in df.columns, f"Colonne {col_diag} absente"
    subset = df.copy()
    subset[f"has_{col_diag}"] = subset[col_diag].notna()
    stats = (
        subset
        .groupby(f"has_{col_diag}")["prix_m2"]
        .agg(count="count", mean="mean", median="median")
        .round(0)
    )
    print(f"\n=== Prix/m² selon présence de {label} ({col_diag}) ===")
    display(stats)
    return stats

stats_dpe = compare_prix_m2_diag(df_clean, "dpe", "DPE")
stats_ges = compare_prix_m2_diag(df_clean, "ges", "GES")


On compare ici le **prix/m² moyen et médian** entre deux groupes d’annonces :
- celles pour lesquelles le diagnostic est **renseigné** (`has_dpe = True` / `has_ges = True`) ;
- celles pour lesquelles il est **absent** (`NaN`).

L’objectif est double :
- mesurer si l’absence d’information énergétique est déjà **sanctionnée** par le marché (prix/m² plus faible) ;
- rappeler que les analyses par classe DPE/GES (A, B, C, F, etc.) portent mécaniquement sur le **segment le mieux documenté**.

En pratique, si les biens sans DPE/GES publié présentent un prix/m² moyen inférieur, cela peut être interprété comme une forme de **signal négatif** : soit le bien est effectivement plus énergivore, soit l’acheteur intègre une incertitude sur la performance énergétique. À l’inverse, une absence d’écart marquée suggère que cette dimension reste encore peu valorisée/prisée à la date du scraping.

In [ ]:
# 2.3.3. Prix/m² moyen par ville

if "ville" in df_explo.columns:
    prix_m2_par_ville = df_explo.groupby("ville")["prix_m2"].mean().sort_values(ascending=False)
    print("Prix moyen au m² par ville :")
    display(prix_m2_par_ville)

    plt.figure()
    prix_m2_par_ville.plot(kind="bar")
    plt.ylabel("Prix moyen au m² (€ / m²)")
    plt.title("Prix moyen au m² par ville (maisons, Bordeaux R10)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Colonne 'ville' absente, impossible de tracer par ville.")
    

Ici, on regroupe les données par `ville` et on calcule le **prix/m² moyen** pour chaque ville, puis on trie les résultats du plus cher au moins cher.

Le tableau permet de voir, ville par ville :
- quelles localités du rayon R10 autour de Bordeaux présentent les niveaux de prix/m² les plus élevés,
- lesquelles sont plus abordables.

Le graphique en barres visualise ces différences :
- axe horizontal : les villes,
- axe vertical : le prix/m² moyen.

À interpréter avec prudence :
- si certaines villes ont peu d’annonces, leur moyenne peut être instable ;
- la présence de **doublons inter-agences non dédoublonnés métier** peut légèrement biaiser certains prix moyens.

Interprétations possibles à ajouter :
- *“Les villes A, B, C se distinguent comme les plus chères en prix/m², ce qui peut s’expliquer par [proximité du centre, environnement privilégié, accès transport, etc.].”*  
- *“À l’inverse, les communes X, Y apparaissent comme plus abordables.”*  

Une extension naturelle est de croiser ces résultats avec :
- la présence de piscines, grands terrains,
- des diagnostics énergétiques favorables,
- le type de localisation (centre-ville vs périphérie).

In [ ]:
# 2.3.4. Relation surface habitable / prix sur df_clean (CLEAN complet)

plt.figure(figsize=(7, 5))

# On garde uniquement les lignes complètes
mask = df_clean["surface_habitable"].notna() & df_clean["prix"].notna()
x = df_clean.loc[mask, "surface_habitable"]
y = df_clean.loc[mask, "prix"]

# Nuage de points : petits points + légère transparence
plt.scatter(x, y, s=12, alpha=0.5)

plt.xlabel("Surface habitable (m²)")
plt.ylabel("Prix (€)")
plt.title("Relation entre surface habitable et prix (maisons, Bordeaux R10 – df_clean)")

# Grille légère pour la lecture
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# Corrélation de Pearson sur le même sous-ensemble
corr = df_clean.loc[mask, ["surface_habitable", "prix"]].corr().iloc[0, 1]
print(f"Corrélation linéaire (Pearson) surface_habitable / prix (df_clean) : {corr:.3f}")


Scatter surface habitable vs prix + corrélation de Pearson  

Le nuage de points (*scatter plot*) représente chaque maison présente dans le fichier **CLEAN** (`df_clean`) :
- **Axe horizontal (x)** : `surface_habitable` (en m²)  
- **Axe vertical (y)** : `prix` (en €)

Chaque point correspond à un bien. Ce graphique permet de voir :
- si, globalement, les maisons plus grandes sont aussi plus chères (ce qui est attendu sur un marché “normal”) ;
- s’il existe des biens :
  - très chers pour une surface relativement réduite (biens haut de gamme, localisation premium, vue exceptionnelle, littoral, etc.) ;
  - ou, au contraire, de grandes surfaces à un prix relativement contenu (biens situés en périphérie ou en zone moins tendue).

En complément, la **corrélation de Pearson** entre `surface_habitable` et `prix` donne une mesure globale de la relation :
- une corrélation proche de **1** indique une **relation linéaire forte** : plus la surface augmente, plus le prix augmente de façon assez régulière ;
- une corrélation **modérée** (par exemple autour de 0,5) montre que la surface joue un rôle important, mais qu’elle ne suffit pas à expliquer le prix (la localisation, la qualité du bien, la taille du terrain, la présence d’une piscine, le DPE, etc. restent déterminants) ;
- une corrélation très faible, proche de **0**, signifierait que la surface seule explique très peu la variation des prix dans ce jeu de données.

Dans notre cas, une interprétation type à formuler pourrait être :
> *« La corrélation positive entre surface habitable et prix confirme que la taille du bien est un déterminant majeur du prix, mais le nuage de points montre aussi une forte dispersion, liée à d’autres facteurs (localisation, équipements, proximité du littoral…). Les points très au-dessus du nuage principal correspondent vraisemblablement à des biens d’exception, typiques du micro-climat immobilier bordelais (par exemple maisons proche du secteur littoral très prisé). »*

> *« Pour ce jeu de données, la corrélation de Pearson est d’environ **X,XXX**, ce qui correspond à une relation [modérée/forte] entre surface et prix. »*  
(remplacer **X,XXX** par la valeur calculée dans le notebook)

In [ ]:
# 2.3.5. Mise en évidence visuelle de quelques outliers en prix/m² sur le scatter surface/prix

# On part du même mask que pour le scatter
mask = df_clean["surface_habitable"].notna() & df_clean["prix"].notna()

# On sélectionne 3 biens avec les prix/m² les plus élevés parmi ceux du scatter
df_tmp = df_clean.loc[mask].copy()
df_tmp = df_tmp.dropna(subset=["prix_m2"])
outliers_scatter = df_tmp.sort_values("prix_m2", ascending=False).head(3)

plt.figure(figsize=(7, 5))

plt.scatter(
    df_tmp["surface_habitable"],
    df_tmp["prix"],
    s=12,
    alpha=0.5
)

# On surligne les outliers avec un marqueur plus gros
plt.scatter(
    outliers_scatter["surface_habitable"],
    outliers_scatter["prix"],
    s=60,
    edgecolor="black"
)

for _, row in outliers_scatter.iterrows():
    label = f"{row.get('ville', '')} ({row['prix_m2']:.0f} €/m²)"
    plt.annotate(
        label,
        (row["surface_habitable"], row["prix"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8
    )

plt.xlabel("Surface habitable (m²)")
plt.ylabel("Prix (€)")
plt.title("Relation surface / prix avec mise en évidence de quelques outliers (df_clean)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


On met ici en évidence quelques biens parmi les **plus chers au m²** directement sur le nuage de points :
- les points “standards” sont affichés en arrière-plan ;
- 2–3 biens très chers au m² sont **mis en avant** avec un marqueur plus gros et une **annotation** (ville, prix/m²).

Cette visualisation permet de :
- confirmer que ces outliers correspondent bien à des biens isolés dans le nuage (prix très élevé pour une surface donnée) ;
- illustrer le lien entre **localisation / caractéristiques premium** et **prix déconnectés** du cœur de marché.

Ce type de figure est utile dans le rapport pour montrer concrètement comment quelques biens de prestige peuvent s’écarter fortement du nuage principal.

### Bilan descriptif global (CLEAN complet)

Sur le fichier CLEAN complet (df_explo), on dispose désormais d’une vision d’ensemble du marché :
- un **niveau de prix au m² typique** situé autour de la médiane calculée plus haut, avec une moyenne légèrement tirée vers le haut par quelques biens très chers ;
- une distribution des **surfaces habitables** concentrée sur des maisons de taille “standard”, mais avec quelques grandes propriétés en périphérie ou sur des zones plus rurales ;
- un **nuage de points surface/prix** qui confirme la relation positive entre taille du bien et prix, tout en montrant une forte dispersion liée à la localisation et aux caractéristiques premium ;
- des **outliers** clairement visibles en prix/m², qui correspondent à des biens d’exception que l’on choisit de conserver pour refléter le marché réel.

Ce socle descriptif sert de point d’appui pour les analyses plus fines qui suivent (effet de la piscine, du terrain, de la localisation, des diagnostics énergétiques, etc.).

In [ ]:
# 2.4. Analyses croisées sur df_clean_iqr

# ============================================================================
# Analyses croisées sur df_clean_iqr (cœur de marché : 5e–95e percentiles prix_m2)
# - Piscine
# - Grand terrain
# - DPE
# - Type de zone (centre / périphérie / littoral…)
# ============================================================================

"""
Les autres équipements (jardin, terrasse, etc.) sont partiellement capturés dans features, mais la variabilité de remplissage
et la granularité des textes rendent plus difficile une quantification robuste de leur effet individuel."""

# 2.4.1. Présence de piscine : création d'un indicateur binaire has_pool
if "features" in df_clean_iqr.columns:
    df_clean_iqr["has_pool"] = df_clean_iqr["features"].str.contains("piscine", case=False, na=False)
else:
    print("⚠ Colonne 'features' absente, impossible de créer has_pool.")

# 2.4.2. Grand terrain : création d'un indicateur binaire grand_terrain
if "surface_terrain" in df_clean_iqr.columns:
    # "Grand terrain" = terrain dans le quart supérieur de la distribution (75e percentile)
    q75_terrain = df_clean_iqr["surface_terrain"].quantile(0.75)
    print(f"Seuil 'grand terrain' (75e percentile) : {q75_terrain:.0f} m²")

    df_clean_iqr["grand_terrain"] = df_clean_iqr["surface_terrain"] >= q75_terrain
else:
    print("⚠ Colonne 'surface_terrain' absente, impossible de créer grand_terrain.")

# 2.4.3. Prix/m² selon présence de piscine
if "has_pool" in df_clean_iqr.columns:
    print("\n=== Prix/m² selon présence de piscine (df_clean_iqr) ===")
    prix_pool = df_clean_iqr.groupby("has_pool")["prix_m2"].describe().round(0)
    display(prix_pool)
else:
    print("⚠ Impossible de calculer prix/m² par piscine (has_pool manquant).")

# 2.4.4. Prix/m² selon taille du terrain (grand_terrain)
if "grand_terrain" in df_clean_iqr.columns:
    print("\n=== Prix/m² selon taille du terrain (grand_terrain, df_clean_iqr) ===")
    prix_terrain = df_clean_iqr.groupby("grand_terrain")["prix_m2"].describe().round(0)
    display(prix_terrain)
else:
    print("⚠ Impossible de calculer prix/m² par taille de terrain (grand_terrain manquant).")

# 2.4.5. Prix/m² et diagnostics énergétiques (DPE)
if "dpe" in df_clean_iqr.columns:
    print("\n=== Prix/m² par classe DPE (df_clean_iqr) ===")
    dpe_stats = (
        df_clean_iqr
        .dropna(subset=["prix_m2", "dpe"])
        .groupby("dpe")["prix_m2"]
        .describe()
        .round(0)
        .sort_index()
    )
    display(dpe_stats)
else:
    print("⚠ Colonne 'dpe' absente, impossible de faire l'analyse DPE.")

# 2.4.6. Prix/m² et type de zone (centre / périphérie / littoral…)

# Dictionnaire de mapping ville -> type de zone (à adapter selon le jeu)
mapping_zone = {
    "Bordeaux": "centre urbain",
    "Mérignac": "périphérie proche",
    "Pessac": "périphérie proche",
    "Talence": "périphérie proche",
    "Gradignan": "périphérie",
    "Eysines": "périphérie",
    "Le Bouscat": "périphérie proche",
    "Pyla-sur-Mer": "littoral premium",
    "La Teste-de-Buch": "littoral",
    "Arcachon": "littoral premium",
    # etc. à compléter si besoin, les villes près du littoral sont données àà titre d'exemple pour une extension du dictionnaire
}

if "ville" in df_clean_iqr.columns:
    df_clean_iqr["type_zone"] = df_clean_iqr["ville"].map(mapping_zone).fillna("autre / non classé")
else:
    print("⚠ Colonne 'ville' absente, impossible de créer type_zone.")

if "type_zone" in df_clean_iqr.columns:
    print("\n=== Prix/m² moyen par type de zone (df_clean_iqr) ===")
    prix_m2_par_zone = (
        df_clean_iqr
        .dropna(subset=["prix_m2"])
        .groupby("type_zone")["prix_m2"]
        .describe()
        .round(0)
        .sort_values("mean", ascending=False)
    )
    display(prix_m2_par_zone)
else:
    print("⚠ Impossible de calculer prix/m² par type de zone (type_zone manquant).")


In [ ]:
# 2.4.7. Calcul de l’impact de la piscine sur le prix/m² (df_clean_iqr)

# Sécurité : vérifier que la colonne has_pool existe bien
assert "has_pool" in df_clean_iqr.columns, "La colonne 'has_pool' est absente. Pense à exécuter la cellule qui la crée."

# Moyenne du prix/m² selon présence de piscine
moyennes_pool = df_clean_iqr.groupby("has_pool")["prix_m2"].mean()

mean_sans_piscine = moyennes_pool.loc[False]
mean_avec_piscine = moyennes_pool.loc[True]

diff_absolue = mean_avec_piscine - mean_sans_piscine
diff_pourcent = (diff_absolue / mean_sans_piscine) * 100

print(f"Prix moyen/m² sans piscine : {mean_sans_piscine:.0f} € / m²")
print(f"Prix moyen/m² avec piscine : {mean_avec_piscine:.0f} € / m²")
print(f"Surcoût absolu moyen      : {diff_absolue:.0f} € / m²")
print(f"Surcoût relatif moyen     : {diff_pourcent:.1f} %")

# Phrase prête à être copiée dans le rapport
phrase = (
    f"Sur le cœur de marché, la présence d'une piscine est associée à un "
    f"surcoût moyen d'environ {diff_pourcent:.1f} % du prix au mètre carré, "
    f"le prix passant d'environ {mean_sans_piscine:.0f} €/m² pour les maisons "
    f"sans piscine à {mean_avec_piscine:.0f} €/m² pour celles qui en disposent."
)

print("\nPhrase pour le rapport :")
print(phrase)


In [ ]:
# 2.4.8. Comparaison des prix/m² par type de zone (df_clean_iqr)

# Centre urbain vs autres zones : différences absolues et relatives

# Sécurité : vérifier la présence de type_zone et prix_m2
assert "type_zone" in df_clean_iqr.columns, "La colonne 'type_zone' est absente. Pense à exécuter la cellule qui la crée."
assert "prix_m2" in df_clean_iqr.columns, "La colonne 'prix_m2' est absente."

# Moyennes de prix_m2 par type de zone
moyennes_zone = df_clean_iqr.groupby("type_zone")["prix_m2"].mean()
print("Prix moyen/m² par type de zone (df_clean_iqr) :")
print(moyennes_zone.round(0))
print()

# Zones que l'on souhaite comparer au centre urbain
zone_ref = "centre urbain"
zones_cibles = ["périphérie proche", "périphérie", "autre / non classé"]

assert zone_ref in moyennes_zone.index, f"La zone de référence '{zone_ref}' n'existe pas dans type_zone."

mean_ref = moyennes_zone.loc[zone_ref]

phrases = []

for z in zones_cibles:
    if z in moyennes_zone.index:
        mean_z = moyennes_zone.loc[z]
        diff_abs = mean_ref - mean_z
        diff_pct = (diff_abs / mean_z) * 100

        print(f"--- {zone_ref} vs {z} ---")
        print(f"Prix moyen/m² {zone_ref:>18} : {mean_ref:8.0f} €/m²")
        print(f"Prix moyen/m² {z:>18} : {mean_z:8.0f} €/m²")
        print(f"Différence absolue            : {diff_abs:8.0f} €/m²")
        print(f"Différence relative           : {diff_pct:8.1f} %\n")

        phrase = (
            f"Par rapport à la zone '{z}', le centre urbain présente un "
            f"prix moyen au mètre carré supérieur d'environ {diff_abs:.0f} €/m², "
            f"soit un écart d'environ {diff_pct:.1f} %. "
            f"On passe ainsi d'environ {mean_z:.0f} €/m² en moyenne dans la zone '{z}' "
            f"à environ {mean_ref:.0f} €/m² dans le centre urbain."
        )
        phrases.append((z, phrase))
    else:
        print(f"⚠ La zone '{z}' n'est pas présente dans les données.\n")

print("\nSynthèse :\n")
for z, p in phrases:
    print(f"- {zone_ref} vs {z} :")
    print(p)
    print()


### Analyses croisées sur le cœur de marché (`df_clean_iqr`)

> Rappel : ces analyses sont faites sur `df_clean_iqr`, le **cœur de marché** (biens dont le prix/m² se situe entre les 5ᵉ et 95ᵉ percentiles).  
> Les outliers sont exclus ici pour stabiliser les moyennes et faciliter les comparaisons.

---

#### 1. Prix/m² selon la présence de piscine

**Tableau (résumé)**  
- Sans piscine : 456 biens, prix/m² moyen ≈ **3 965 €/m²**, médiane ≈ **3 912 €/m²**  
- Avec piscine : 82 biens, prix/m² moyen ≈ **4 183 €/m²**, médiane ≈ **4 125 €/m²**

**Lecture et interprétation**
- Les maisons **avec piscine** ont en moyenne un prix au mètre carré **plus élevé** (≈ +200 €/m²).
- La médiane suit la même tendance : la piscine est associée à un **positionnement plus haut dans la distribution des prix/m²**, pas seulement à quelques cas isolés.
- L’écart-type est comparable : l’effet piscine décale le niveau moyen sans créer une catégorie totalement à part.
- Le nombre d’observations avec piscine (82) reste **nettement plus faible** que sans piscine.

En résumé : sur le cœur de marché, la présence d’une piscine est associée à un **surcoût moyen d’environ 5,5 %** du prix au mètre carré, le prix passant d’environ **3 965 €/m²** à **4 183 €/m²**.

---

#### 2. Prix/m² selon la taille du terrain (`grand_terrain`)

**Définition**  
- Seuil “grand terrain” : 75ᵉ percentile de `surface_terrain`, soit ≈ **740 m²**.  
- `grand_terrain = True` : terrains ≥ 740 m².  
- `grand_terrain = False` : terrains < 740 m².

**Tableau (résumé)**  
- Terrain “non grand” : 425 biens, prix/m² moyen ≈ **4 068 €/m²**, médiane ≈ **4 000 €/m²**  
- “Grand terrain” : 113 biens, prix/m² moyen ≈ **3 737 €/m²**, médiane ≈ **3 662 €/m²**

**Lecture et interprétation**
- De manière contre-intuitive, les biens avec un **grand terrain** ont un prix/m² **plus faible**.
- Ce résultat est classique :
  - les grandes parcelles se trouvent plus souvent en **périphérie ou zones moins denses**, au prix/m² plus bas ;
  - les terrains plus petits sont fréquents en **zone urbaine ou périurbaine recherchée**, au foncier plus cher.
- L’écart de médiane (≈ 4 000 vs ≈ 3 662 €/m²) montre que ce n’est pas qu’un effet d’outliers.

En résumé : **plus de terrain ne rime pas forcément avec prix/m² plus élevé** ; ici, les grands terrains sont plutôt associés à des zones moins tendues, avec un prix/m² inférieur.

---

#### 3. Prix/m² moyen par type de zone

**Rappel du mapping (simplifié)**  
- `centre urbain` : Bordeaux  
- `périphérie proche` : Mérignac, Pessac, Talence, Le Bouscat…  
- `périphérie` : Gradignan, Eysines…  
- `littoral premium` / `littoral` : Arcachon, Pyla-sur-Mer, La Teste-de-Buch… (éventuellement filtrés par l’IQR si très extrêmes)  
- `autre / non classé` : autres communes ou villes non mappées.

**Tableau (résumé, cœur de marché)**  
- Centre urbain : 140 biens, ≈ **4 499 €/m²**  
- Périphérie proche : 94 biens, ≈ **4 079 €/m²**  
- Autre / non classé : 281 biens, ≈ **3 748 €/m²**  
- Périphérie : 23 biens, ≈ **3 682 €/m²**

**Écarts par rapport au centre urbain**
- vs périphérie proche : surcoût ≈ **+420 €/m²** (**+10,3 %**)  
- vs périphérie : surcoût ≈ **+817 €/m²** (**+22,2 %**)  
- vs « autre / non classé » : surcoût ≈ **+750 €/m²** (**+20,0 %**)

**Lecture et interprétation**
- On retrouve un **gradient géographique classique** :
  - le **centre urbain** (Bordeaux) est le plus cher ;
  - la **périphérie proche** reste chère mais ~10 % en dessous ;
  - la **périphérie** et les communes **« autre / non classé »** sont ~20–22 % moins chères au m².
- Cela reflète la structure du marché bordelais :
  - forte attractivité du centre et de la première couronne ;
  - prix plus modérés en s’éloignant, avec surfaces souvent plus grandes.

En résumé : le **type de zone** est un déterminant majeur du prix au m². À périmètre constant (maisons, cœur de marché bordelais), habiter dans le centre urbain coûte de l’ordre de **10 à 20 % plus cher au m²** que dans la 1ʳᵉ ou 2ᵉ couronne.

In [ ]:
# 2.5. Analyse conjointe DPE / GES sur df_clean_iqr (coeur de marché)

# ============================================================================
# Analyse conjointe DPE / GES sur df_clean_iqr (coeur de marché)
# ============================================================================

# 2.5.1. Sécurité : vérifier la présence des colonnes nécessaires
for col in ["dpe", "ges", "prix_m2"]:
    assert col in df_clean_iqr.columns, f"Colonne manquante : {col}"

# 2.5.2. On ne garde que les lignes avec DPE, GES et prix_m2 renseignés
df_ener = df_clean_iqr.dropna(subset=["dpe", "ges", "prix_m2"]).copy()
print(f"Nombre d'annonces avec DPE + GES + prix_m2 disponibles : {len(df_ener)}")

# 2.5.3. Prix/m² moyen par DPE (rappel)
print("\n=== Prix/m² moyen par classe DPE (df_clean_iqr) ===")
mean_dpe = df_ener.groupby("dpe")["prix_m2"].mean().round(0)
display(mean_dpe)

# 2.5.4. Prix/m² moyen par GES
print("\n=== Prix/m² moyen par classe GES (df_clean_iqr) ===")
mean_ges = df_ener.groupby("ges")["prix_m2"].mean().round(0)
display(mean_ges)

# 2.5.5. Tableau croisé DPE x GES : prix/m² moyen
print("\n=== Prix/m² moyen par couple (DPE, GES) ===")
pivot_dpe_ges = (
    df_ener
    .pivot_table(
        index="dpe",
        columns="ges",
        values="prix_m2",
        aggfunc="mean"
    )
    .round(0)
)

display(pivot_dpe_ges)

# 2.5.6. Stats descriptives détaillées par couple (DPE, GES)
print("\n=== Statistiques descriptives par couple (DPE, GES) ===")
dpe_ges_stats = (
    df_ener
    .groupby(["dpe", "ges"])["prix_m2"]
    .describe()
    .round(0)
)

display(dpe_ges_stats)


### Analyse conjointe DPE / GES et niveau de prix au m² (cœur de marché)

> Périmètre : uniquement les biens du **cœur de marché** (`df_clean_iqr`) pour lesquels **DPE, GES et prix/m²** sont tous renseignés (≈ 489 annonces).

---

#### 1. Prix/m² moyen par classe DPE

Prix moyen au m² par classe DPE (approx.) :
- **A** : ≈ 4 381 €/m²  
- **B** : ≈ 4 203 €/m²  
- **C** : ≈ 3 937 €/m²  
- **D** : ≈ 4 066 €/m²  
- **E** : ≈ 3 657 €/m²  
- **F** : ≈ 3 315 €/m²  
- **G** : ≈ 4 269 €/m²  

**Lecture**
- Tendance globale cohérente :  
  - les classes **A, B, D** se situent **au-dessus** de la classe **E** ;  
  - la classe **F** est nettement **en dessous** ;  
  - la classe **C** est intermédiaire.
- En ordre de grandeur, les classes **A/B/D** présentent un prix/m² **environ 10 à 20 % plus élevé** que la classe **E**, alors que les biens en **F** sont en moyenne autour de **−10 %** sous E.
- La classe **G** affiche un prix/m² élevé, mais avec **très peu d’observations**, ce qui appelle à la prudence.

En résumé, même sur le cœur de marché, **les biens les mieux classés énergétiquement sont survalorisés en prix au m²**, ce qui reflète l’importance croissante de la performance énergétique.

---

#### 2. Prix/m² moyen par classe GES

Prix moyen au m² par classe GES (approx.) :
- **A** : ≈ 4 067 €/m²  
- **B** : ≈ 4 154 €/m²  
- **C** : ≈ 3 975 €/m²  
- **D** : ≈ 3 990 €/m²  
- **E** : ≈ 3 647 €/m²  
- **F** : ≈ 3 096 €/m²  
- **G** : ≈ 4 533 €/m²  

**Lecture**
- Structure proche du DPE :
  - les classes **A, B, C, D** se situent autour de **4 000 €/m²** ;
  - les classes **E** et surtout **F** sont significativement plus basses ;
  - la classe **G** ressort avec un prix moyen élevé **mais sur très peu de biens**.
- Les biens en **GES F** semblent **les plus pénalisés** : leur prix/m² moyen est nettement inférieur à celui des classes A–D.

Cela suggère que les enjeux **d’émissions de gaz à effet de serre** sont eux aussi pris en compte dans la formation des prix.

---

#### 3. Analyse croisée DPE × GES : couples énergétiques et prix/m²

Le tableau croisé DPE × GES montre le **prix/m² moyen par couple de classes**. Quelques exemples marquants :
- **DPE A – GES A** : ≈ **4 381 €/m²**  
  → Biens globalement performants (énergie + GES) avec un niveau de prix élevé.
- **DPE B – GES B** : ≈ **4 466 €/m²**  
  → Très bon couple énergétique, parmi les plus chers (effectif réduit).
- **DPE C – GES C** : ≈ **3 983 €/m²**  
  → Profil intermédiaire, légèrement inférieur aux meilleurs couples.
- **DPE D – GES B/C/D** : autour de **4 000–4 200 €/m²**  
  → Biens à performance correcte, bien valorisés.
- **DPE E – GES C/D/E** : ≈ **3 600–3 700 €/m²**  
  → Segment médian/faible sur le plan énergétique, en retrait par rapport aux meilleurs couples.
- **DPE F – GES F** : ≈ **3 096 €/m²**  
  → L’un des couples les plus défavorables (biens très énergivores), avec un **prix/m² nettement inférieur** à la moyenne du cœur de marché.
- **DPE G – GES G** : ≈ **4 533 €/m²**  
  → Cas atypique : très mauvais diagnostics mais prix/m² très élevé, probablement lié à des biens **très spécifiques** (localisation premium, bien de prestige). **Effectif très faible**.

**Lecture globale**
- Les couples **A/A**, **B/B**, **C/C** ou **D/B/C/D** se situent dans le haut de la distribution, ce qui confirme que les bons diagnostics sont associés à des biens mieux valorisés.
- À l’inverse, les couples dégradés type **F/F** sont nettement sous la moyenne (biens moins attractifs ou nécessitant des travaux).
- Les couples “paradoxaux” (ex : G/G) montrent que pour quelques biens rares et très bien situés, la localisation et le prestige peuvent **dominer** l’effet énergétique.

---

#### 4. Conclusion intermédiaire : DPE, GES et valorisation des biens

L’analyse conjointe DPE / GES montre que :
- Les diagnostics énergétiques **ne sont pas neutres** : les biens bien classés (A, B, C, D) se situent en moyenne **au-dessus** des classes E / F en niveau de prix/m².
- Les classes les plus dégradées (DPE F, GES F) apparaissent **globalement pénalisées** en prix/m² dans le cœur de marché.
- Quelques exceptions (par exemple G/G, très peu d’observations) rappellent que la **localisation exceptionnelle** ou le caractère très premium d’un bien peut compenser un mauvais diagnostic énergétique.

Dans le contexte bordelais – marché tendu, attractif, avec un “micro-climat” immobilier – la performance énergétique joue donc un **rôle réel mais modulé** par d’autres facteurs majeurs : localisation (centre vs périphérie vs littoral), type de bien, équipements (piscine, terrain, etc.).

### Bilan des analyses croisées sur le cœur de marché

Sur le cœur de marché (df_clean_iqr), les analyses croisées mettent en évidence plusieurs effets structurants :
- la **présence d’une piscine** est associée à un surcoût moyen en prix/m², cohérent avec un positionnement plus premium des biens ;
- les **grands terrains** sont paradoxalement liés à un prix/m² plus faible, ce qui renvoie à des localisations plus périphériques, où le foncier est moins cher ;
- le **type de zone** (centre urbain, périphérie proche, périphérie, autres communes) joue un rôle déterminant, avec un gradient clair de prix au m² au profit du centre de Bordeaux.

Ces résultats confirment que, à prix/m² égal, le marché intègre simultanément des critères de localisation, de confort (piscine, terrain) et de contexte urbain, avant même de considérer la performance énergétique détaillée (DPE/GES).

In [ ]:
# 2.6. Audit des valeurs manquantes (NaN) dans df_clean

# ============================================================================
# Audit des valeurs manquantes (NaN) dans df_clean
# ============================================================================

# Colonnes que l'on souhaite auditer
cols_auditer = ["surface_terrain", "pieces", "dpe", "ges", "features"]

total = len(df_clean)
print(f"Nombre total de lignes dans df_clean : {total}\n")

resultats = []

for col in cols_auditer:
    if col in df_clean.columns:
        n_non_null = df_clean[col].notna().sum()
        n_nan = df_clean[col].isna().sum()
        pct_nan = (n_nan / total) * 100 if total > 0 else 0
        resultats.append({
            "colonne": col,
            "non_null": n_non_null,
            "nan": n_nan,
            "pct_nan": round(pct_nan, 1)
        })
    else:
        print(f"⚠ Colonne '{col}' absente de df_clean")

# Mise en forme dans un DataFrame pour affichage clair
df_nan_audit = pd.DataFrame(resultats)[["colonne", "non_null", "nan", "pct_nan"]]

print("Résumé des valeurs manquantes :")
display(df_nan_audit)

#### Focus sur les valeurs manquantes : `surface_terrain` et `features`

Sur le fichier CLEAN (`df_clean`, 598 annonces), deux variables importantes présentent un taux de valeurs manquantes non négligeable :
- `surface_terrain` : 499 valeurs renseignées, **99 manquantes**, soit ≈ **16,6 %** ;  
- `features` (équipements : piscine, jardin, terrasse, parking, etc.) : 537 valeurs renseignées, **61 manquantes**, soit ≈ **10,2 %**.

---

##### 1. `surface_terrain` : 16,6 % de valeurs manquantes

La surface de terrain n’est pas renseignée dans environ **un bien sur six**.

Pistes d’explication :
- certains biens (maisons de ville, échoppes, bâti mitoyen) disposent d’un jardin ou patio très réduit, jugé peu pertinent à détailler en m² ;
- dans d’autres cas, l’agence ou le vendeur peut **connaître le terrain mais ne pas le saisir** dans l’annonce (qualité de saisie, hétérogénéité des pratiques, etc.) ;
- pour quelques biens, l’information peut être réellement inconnue ou difficile à exprimer (parcelles atypiques, divisions, copropriété…).

Conséquences pour l’analyse :
- les statistiques impliquant `surface_terrain` (ex. “grand terrain” vs standard) ne portent que sur les **83,4 % d’annonces renseignées** ;
- on ne sait pas, à partir des données seules, si “terrain non renseigné” signifie **terrain nul**, **très petit** ou potentiellement simplement **information omise**.

Cela montre que la variable “terrain” est **moins standardisée** que le prix ou la surface habitable, alors qu’elle joue un rôle réel dans la valorisation d’une maison. Cependant, avec la mise sous tension du marché, cette information devient un véritable atout commercial.

---

##### 2. `features` : 10,2 % de valeurs manquantes

La colonne `features` agrège des informations qualitatives comme :
- présence d’une **piscine**,
- **jardin**,
- **terrasse / balcon**,
- **parking** ou **garage**, etc.

Environ **une annonce sur dix** ne contient aucune information dans ce champ (`NaN`).

Interprétations possibles :
- certains biens n’ont effectivement **aucun équipement particulier** ;
- mais dans d’autres cas, les équipements existent peut-être, sans être codés dans `features` (annonce peu détaillée, scraping incomplet si les icônes ne sont pas présentes ou lisibles).

Conséquences :
- les comparaisons **avec vs sans piscine**, ou **grand terrain vs terrain standard**, reposent de fait sur le sous-ensemble d’annonces **bien renseignées en `features`** ;
- on peut donc **sous-estimer** certains effets si des équipements ne sont pas systématiquement déclarés.

---

##### 3. Lecture globale : qualité d’annonce et prudence d’interprétation

Ces taux de manquants (≈ 16,6 % pour `surface_terrain` et ≈ 10,2 % pour `features`) ne remettent pas en cause la faisabilité de l’analyse, mais révèlent :
- une **hétérogénéité de la qualité de saisie** entre annonces (certaines très complètes, d’autres minimalistes) ;
- un risque de **biais de sélection** dès qu’une analyse est conditionnée à la présence de ces variables (segment “bien documenté” seulement).

Dans le rapport, il est donc important de préciser que :
- les résultats sur les effets du terrain, des piscines et des équipements doivent être lus comme une **tendance sur les annonces complètes** ;
- les valeurs manquantes sont aussi un **symptôme métier** : la façon dont les biens sont présentés au marché (plus ou moins transparents, plus ou moins détaillés) est déjà une information en soi.

## Conclusion générale

Ce projet avait pour objectif de comprendre **comment le prix au mètre carré varie en fonction de la localisation, de la surface et des caractéristiques des maisons** autour de Bordeaux, à partir de données issues de web scraping.

Sur le plan méthodologique, nous avons mis en place un pipeline complet et reproductible :
- collecte automatisée des annonces via un **scraper dédié** (EtreProprio.com, maisons, rayon 10 km autour de Bordeaux) ;  
- diagnostic du **fichier brut** (`df_raw`) : structure, types, valeurs manquantes, doublons techniques et début d’analyse des doublons “métier” multi-agences ;  
- construction d’un fichier **CLEAN** (`df_clean` / `df_final`) : typage numérique, suppression des doublons stricts et d’URL, filtrage des annonces inexploitables (prix nul ou surface trop faible), création de la variable clé `prix_m2` ;  
- définition d’un **cœur de marché** (`df_clean_iqr`), basé sur les percentiles 5–95 % de `prix_m2`, utilisé pour stabiliser certaines moyennes et comparaisons ;  
- enrichissement **géographique** par géocodage (`ville`, `cp` → `lat`, `lon`) en vue de futures visualisations cartographiques (folium) et d’un dashboard Streamlit.

Sur le plan économique, plusieurs enseignements ressortent :
- Le niveau “typique” de prix au m² se situe autour de la **médiane** calculée sur le fichier CLEAN, la **moyenne** étant légèrement plus élevée sous l’effet de quelques biens de prestige. Cela confirme que le marché est marqué par une queue haute de prix, concentrée sur certains segments spécifiques.  
- La **localisation** est un déterminant majeur : à périmètre constant (maisons), habiter dans le **centre urbain bordelais** se paie de l’ordre de **10 à 20 % plus cher au m²** que dans la périphérie proche ou les autres communes du périmètre. Le type de zone (centre, périphérie proche, périphérie, autres) structure clairement le niveau de prix.  
- Les **caractéristiques des biens** jouent également un rôle important :
  - la présence d’une **piscine** est associée à un **surcoût moyen en prix au m²** sur le cœur de marché ;  
  - les **grands terrains** sont le plus souvent situés en zones moins denses et s’accompagnent paradoxalement d’un prix/m² plus faible, en cohérence avec un foncier moins cher en périphérie.  
- Les diagnostics **DPE / GES** ne sont pas neutres : les classes A/B/C/D sont globalement mieux valorisées que les classes E/F, et certains couples énergétiques dégradés (par exemple F/F) se situent nettement sous la moyenne du cœur de marché. Quelques cas atypiques (mauvais diagnostics mais prix très élevé) rappellent néanmoins que, pour certains biens de prestige, la **localisation** peut dominer l’effet énergétique.

L’analyse des valeurs manquantes montre enfin que :
- les informations “de base” (prix, surface habitable, ville, code postal) sont bien renseignées, ce qui permet de répondre à la problématique principale ;  
- en revanche, des variables plus qualitatives comme `surface_terrain`, `features` ou les diagnostics `dpe` / `ges` présentent entre **10 % et 17 %** de valeurs manquantes, ce qui introduit un **biais de sélection** : les analyses les plus fines portent en priorité sur le segment des annonces les mieux documentées.

Ce travail pose ainsi les fondations d’un **outil d’analyse reproductible** (scripts de scraping, de nettoyage, d’analyse) et ouvre plusieurs pistes de prolongement :
- enrichir la **géolocalisation** (niveau adresse ou quartier) pour construire des cartes plus fines et des analyses par micro-secteur ;  
- étendre le périmètre à d’autres **zones géographiques** ou à plusieurs plateformes pour mieux couvrir le marché ;  
- intégrer un **historique temporel** afin de suivre l’évolution des prix et la dynamique du marché ;  
- développer des **modèles prédictifs de prix/m²** à partir d’un ensemble élargi de variables (localisation fine, surface, équipements, DPE/GES, etc.), en s’appuyant sur la base CLEAN constituée dans ce projet.

En résumé, même avec les limites inhérentes aux données d’annonces en ligne (snapshot, site unique, complétude variable), ce projet montre qu’un pipeline Python bien structuré permet déjà de dégager des **tendances claires** sur le marché des maisons autour de Bordeaux, et de préparer des analyses plus avancées dans un cadre de data analytics professionnel.

Au regard de la problématique initiale, on peut résumer ainsi :
- le **prix au mètre carré** augmente nettement avec la **centralité géographique** : habiter dans le centre de Bordeaux revient en moyenne 10 à 20 % plus cher au m² que dans la périphérie proche ou les communes voisines ;
- la **surface** et le **terrain** jouent surtout via la localisation : les grandes surfaces et les grands terrains sont plus fréquents en périphérie, avec un prix/m² inférieur malgré un prix total parfois élevé ;
- certaines **caractéristiques des biens** (piscine, diagnostics énergétiques favorables) sont associées à un surcoût mesurable en prix/m², tandis que les biens très énergivores sont globalement pénalisés, sauf cas très spécifiques de biens de prestige.